# Black-box bottom-k sources and robust targets

Generate target-to-least-damaging sources and source-to-least-robust targets using the shared transfer-view configuration.

## Imports, utilities, and config

Default model root: `logs/ByModel`. Default output path: `outputs/blackbox_transfer_views`.

In [6]:
from pathlib import Path
import sys

try:
    from analytics.paths import find_project_root, output_dir
except ModuleNotFoundError:
    repo_root = Path.cwd()
    while repo_root != repo_root.parent and not (repo_root / "src").exists():
        repo_root = repo_root.parent
    sys.path.insert(0, str(repo_root / "src"))
    from analytics.paths import find_project_root, output_dir

from analytics.attacks.bottom_sources_targets import (
    build_bottom_sources_by_target,
    build_bottom_targets_by_source,
)
from analytics.attacks.transfer_role_tools import (
    describe_config,
    get_template,
    list_templates,
    load_or_build_pair_metrics,
    resolve_metric,
    write_outputs,
)

ROOT = find_project_root()
TRAINING_ROOT = ROOT / "logs" / "ByModel"
PAIR_METRICS_CACHE = output_dir(ROOT, "attack_transfer_role_tables") / "pair_metrics.csv"
OUT_DIR = output_dir(ROOT, "blackbox_transfer_views")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEMPLATE_NAME = "architecture_mean"

CONFIG = get_template(
    TEMPLATE_NAME,
    source_grain="architecture",
    target_grain="architecture",
    comparison_mode="inter_architecture",
    metric="Transfer ASR",
    top_k=3,
    variant_aggregate_mode="mean",
    architecture_aggregate_mode="mean",
    split_ansatz_as_architecture=False,
    model_blacklist=tuple(),
    blacklist_contains=tuple(),
)

display(describe_config(CONFIG))

,value
source_grain,architecture
target_grain,architecture
comparison_mode,inter_architecture
metric,Transfer ASR
top_k,3
variant_aggregate_mode,mean
architecture_aggregate_mode,mean
split_ansatz_as_architecture,False
include_no_entangling_as_ansatz,True
model_blacklist,()


In [2]:
display(list_templates())

,template,source_grain,target_grain,comparison_mode,metric,top_k,variant_aggregate_mode,architecture_aggregate_mode,split_ansatz_as_architecture,include_no_entangling_as_ansatz,model_blacklist,blacklist_contains,source_family_filter,target_family_filter,source_architecture_filter,target_architecture_filter,source_contains,target_contains
0,variant_full,variant,variant,inter_variant,Transfer ASR,3,mean,mean,False,True,(),(),None,None,None,None,None,None
1,architecture_mean,architecture,architecture,inter_architecture,Transfer ASR,3,mean,mean,False,True,(),(),None,None,None,None,None,None
2,architecture_best_damage,architecture,architecture,inter_architecture,Transfer ASR,3,best_damage,mean,False,True,(),(),None,None,None,None,None,None
3,architecture_best_robust,architecture,architecture,inter_architecture,Transfer ASR,3,best_robust,mean,False,True,(),(),None,None,None,None,None,None
4,architecture_ansatz_mean,architecture,architecture,inter_architecture,Transfer ASR,3,mean,mean,True,True,(),(),None,None,None,None,None,None
5,family_mean_architecture_mean_variant,family,family,inter_variant,Transfer ASR,3,mean,mean,False,True,(),(),None,None,None,None,None,None
6,family_mean_architecture_best_damage_variant,family,family,inter_variant,Transfer ASR,3,best_damage,mean,False,True,(),(),None,None,None,None,None,None
7,family_mean_architecture_best_robust_variant,family,family,inter_variant,Transfer ASR,3,best_robust,mean,False,True,(),(),None,None,None,None,None,None
8,inter_family_architecture_mean,architecture,architecture,inter_model,Transfer ASR,3,mean,mean,False,True,(),(),None,None,None,None,None,None
9,top_variants_vs_target_architecture,variant,architecture,inter_architecture,Transfer ASR,3,mean,mean,False,True,(),(),None,None,None,None,None,None


## Crawl and find models/artifacts

Inspect discovered model anchors and the cached pair-metrics artifact before loading the table.

In [3]:
model_dirs = sorted(p.parent for p in TRAINING_ROOT.rglob(".model"))
print(f"Model directories found: {len(model_dirs):,}")
print(f"Pair metrics cache: {PAIR_METRICS_CACHE}")
print(f"Pair metrics cache exists: {PAIR_METRICS_CACHE.exists()}")
model_dirs[:10]

Model directories found: 51
Pair metrics cache: C:\Users\fwilh\Desktop\QB-PIER\outputs\attack_transfer_role_tables\pair_metrics.csv
Pair metrics cache exists: True


[WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/logs/ByModel/Classical/convnet/conv_channels=[32,64],fc_sizes=[128]'),
 WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/logs/ByModel/Classical/convnet/conv_channels=[8],fc_sizes=[10]'),
 WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/logs/ByModel/Classical/convnet/conv_channels=[8],fc_sizes=[32]'),
 WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/logs/ByModel/Classical/lenet'),
 WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/logs/ByModel/Classical/mlp/hidden_sizes=[1024, 512, 64, 64]'),
 WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/logs/ByModel/Classical/mlp/hidden_sizes=[128,64]'),
 WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/logs/ByModel/Classical/mlp/hidden_sizes=[32,16]'),
 WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/logs/ByModel/Classical/mlp/hidden_sizes=[4,2]'),
 WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/logs/ByModel/Classical/mlp/hidden_sizes=[4,4]'),
 WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/logs/ByModel/Classical/mlp/hidden_sizes=[64,32]')]

## Load models or prepare data

Load the pair metrics after inspecting the discovered artifacts.

In [7]:
pair_df = load_or_build_pair_metrics(
    pair_metrics_cache=PAIR_METRICS_CACHE,
    rebuild=False,
)
print(f"Loaded pair metrics: {pair_df.shape[0]:,} rows x {pair_df.shape[1]:,} columns")
pair_df.head()

Loaded pair metrics: 1,089 rows x 24 columns


,Source Model,Source Family,Source Architecture,Source Variant,Target Model,Target Family,Target Architecture,Target Variant,WB/BB Flag,Is Self Attack,...,Flip Rate,ASR,Transfer Robust Accuracy,Transfer ASR,Transfer Gap,True-class Confidence,Mean Confidence Drop,Logit Margin,Mean Margin Drop,#Seeds Aggregated
0,Classical/CNN/LeNet,Classical,CNN,LeNet,Classical/CNN/LeNet,Classical,CNN,LeNet,WB,True,...,0.927036,0.938681,NaN,NaN,NaN,NaN,0.913137,NaN,36.846454,2
1,Classical/CNN/LeNet,Classical,CNN,LeNet,"Classical/CNN/conv_channels=[32,64], fc_sizes=...",Classical,CNN,"conv_channels=[32,64], fc_sizes=[128]",BB,False,...,0.724309,0.727893,0.268981,0.727893,0.210789,NaN,0.717797,NaN,13.182026,2
2,Classical/CNN/LeNet,Classical,CNN,LeNet,"Classical/CNN/conv_channels=[8], fc_sizes=[10]",Classical,CNN,"conv_channels=[8], fc_sizes=[10]",BB,False,...,0.835438,0.843873,0.151150,0.843873,0.094808,NaN,0.795750,NaN,28.317362,2
3,Classical/CNN/LeNet,Classical,CNN,LeNet,"Classical/CNN/conv_channels=[8], fc_sizes=[32]",Classical,CNN,"conv_channels=[8], fc_sizes=[32]",BB,False,...,0.833468,0.838785,0.157938,0.838785,0.099897,NaN,0.809189,NaN,31.474350,2
4,Classical/CNN/LeNet,Classical,CNN,LeNet,"Classical/MLP/hidden_sizes=[1024,512,64,64]",Classical,MLP,"hidden_sizes=[1024,512,64,64]",BB,False,...,0.719919,0.723510,0.273879,0.723510,0.215171,NaN,0.716921,NaN,14.138173,2


## Generate the figure/table

Bottom best-attack sources are least damaging sources per target. Bottom robust targets are most damaged / least robust targets per source.

In [8]:
bottom_sources_wide, bottom_sources_long = build_bottom_sources_by_target(pair_df, CONFIG)
print("Bottom attack sources by target")
display(bottom_sources_wide)

Bottom attack sources by target


,target,least damaging 1 source (Transfer ASR),least damaging 2 source (Transfer ASR),least damaging 3 source (Transfer ASR)
0,Classical/CNN,Hybrid/HQE (51.0%),Hybrid/HVQL (64.3%),Hybrid/HVQM (64.3%)
1,Classical/MLP,Hybrid/HQE (55.8%),Hybrid/HVQL (70.9%),Hybrid/HVQM (71.5%)
2,Classical/MLPMixer,Hybrid/HQE (57.7%),Hybrid/HVQL (72.6%),Classical/ResNet18 (73.0%)
3,Classical/ResNet18,Hybrid/HQE (64.2%),Classical/MLP (72.9%),Hybrid/HQuanv (74.1%)
4,Hybrid/HPVQL,Hybrid/HQE (44.6%),Classical/MLPMixer (65.1%),Hybrid/HVQM (65.5%)
5,Hybrid/HQE,Hybrid/HVQL (63.3%),Hybrid/HVQM (64.2%),Hybrid/HPVQL (65.5%)
6,Hybrid/HQuanv,Hybrid/HQE (46.7%),Hybrid/HVQL (57.7%),Hybrid/HVQM (61.0%)
7,Hybrid/HVQL,Hybrid/HQE (48.1%),Classical/MLPMixer (66.8%),Hybrid/HVQM (67.1%)
8,Hybrid/HVQM,Hybrid/HQE (43.8%),Classical/MLPMixer (61.8%),Hybrid/HPVQL (62.6%)
9,Quantum/VQM,Hybrid/HQE (43.1%),Hybrid/HVQL (52.8%),Hybrid/HPVQL (53.7%)


In [9]:
bottom_targets_wide, bottom_targets_long = build_bottom_targets_by_source(pair_df, CONFIG)
print("Bottom robust targets by source")
display(bottom_targets_wide)

Bottom robust targets by source


,source,top 1 damaged target (Transfer ASR),top 2 damaged target (Transfer ASR),top 3 damaged target (Transfer ASR)
0,Classical/CNN,Classical/MLP (83.7%),Hybrid/HQE (80.6%),Hybrid/HQuanv (79.4%)
1,Classical/MLP,Hybrid/HQE (80.5%),Hybrid/HQuanv (80.3%),Classical/CNN (78.1%)
2,Classical/MLPMixer,Classical/ResNet18 (76.1%),Classical/MLP (73.2%),Hybrid/HQE (67.3%)
3,Classical/ResNet18,Classical/MLP (75.0%),Classical/MLPMixer (73.0%),Hybrid/HVQL (70.9%)
4,Hybrid/HPVQL,Classical/ResNet18 (76.3%),Classical/MLPMixer (73.1%),Hybrid/HVQL (72.5%)
5,Hybrid/HQE,Classical/ResNet18 (64.2%),Classical/MLPMixer (57.7%),Classical/MLP (55.8%)
6,Hybrid/HQuanv,Classical/MLP (86.4%),Hybrid/HQE (85.0%),Classical/CNN (80.7%)
7,Hybrid/HVQL,Classical/ResNet18 (76.1%),Classical/MLPMixer (72.6%),Classical/MLP (70.9%)
8,Hybrid/HVQM,Classical/ResNet18 (75.2%),Classical/MLPMixer (73.0%),Classical/MLP (71.5%)
9,Quantum/VQM,Classical/ResNet18 (76.1%),Classical/MLPMixer (74.6%),Classical/MLP (73.1%)


In [10]:
metric_name = resolve_metric(CONFIG.metric).replace(" ", "_")
prefix = f"bottomk_{CONFIG.source_grain}_to_{CONFIG.target_grain}_{CONFIG.comparison_mode}_{metric_name}"
outputs = write_outputs(
    OUT_DIR,
    prefix=prefix,
    top_sources_wide=bottom_sources_wide,
    top_sources_long=bottom_sources_long,
    top_targets_wide=bottom_targets_wide,
    top_targets_long=bottom_targets_long,
)
outputs

{'top_sources_wide': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/blackbox_transfer_views/bottomk_architecture_to_architecture_inter_architecture_Transfer_ASR_top_sources_wide.csv'),
 'top_sources_wide_latex': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/blackbox_transfer_views/bottomk_architecture_to_architecture_inter_architecture_Transfer_ASR_top_sources_wide.tex'),
 'top_sources_long': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/blackbox_transfer_views/bottomk_architecture_to_architecture_inter_architecture_Transfer_ASR_top_sources_long.csv'),
 'top_targets_wide': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/blackbox_transfer_views/bottomk_architecture_to_architecture_inter_architecture_Transfer_ASR_top_targets_wide.csv'),
 'top_targets_wide_latex': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/blackbox_transfer_views/bottomk_architecture_to_architecture_inter_architecture_Transfer_ASR_top_targets_wide.tex'),
 'top_targets_long': WindowsPath('C:/U